In [ ]:
!pip install boto3 pandas numpy scikit-learn matplotlib seaborn -q
print('All libraries installed!')

All libraries installed!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving olist_order_payments_dataset.csv to olist_order_payments_dataset (1).csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset (1).csv
Saving olist_orders_dataset.csv to olist_orders_dataset (1).csv
Saving olist_products_dataset.csv to olist_products_dataset (1).csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset (1).csv
Saving product_category_name_translation.csv to product_category_name_translation (1).csv
Saving olist_customers_dataset.csv to olist_customers_dataset (1).csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset (1).csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset (1).csv


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

orders    = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
items     = pd.read_csv('olist_order_items_dataset.csv')
products  = pd.read_csv('olist_products_dataset.csv')
payments  = pd.read_csv('olist_order_payments_dataset.csv')
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')
sellers   = pd.read_csv('olist_sellers_dataset.csv')
geo       = pd.read_csv('olist_geolocation_dataset.csv')
trans     = pd.read_csv('product_category_name_translation.csv')

print('Orders shape:', orders.shape)
print('Customers shape:', customers.shape)

Orders shape: (99441, 8)
Customers shape: (99441, 5)


In [ ]:
# Convert dates
for col in ['order_purchase_timestamp','order_delivered_customer_date',
            'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Keep only delivered orders
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# Calculate delivery delay
orders_delivered['delivery_delay_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_estimated_delivery_date']
).dt.days

# Build master table
df = orders_delivered.merge(customers, on='customer_id', how='left')
df = df.merge(
    payments.groupby('order_id').agg(
        payment_value=('payment_value','sum'),
        payment_type =('payment_type','first')
    ).reset_index(), on='order_id', how='left')
df = df.merge(
    reviews.groupby('order_id')['review_score'].mean().reset_index(),
    on='order_id', how='left')

print('Master table:', df.shape)

Master table: (96478, 16)


In [ ]:
ref_date = orders_delivered['order_purchase_timestamp'].max()

customer_features = df.groupby('customer_unique_id').agg(
    total_orders        = ('order_id',           'count'),
    total_revenue       = ('payment_value',       'sum'),
    avg_order_value     = ('payment_value',       'mean'),
    avg_review_score    = ('review_score',        'mean'),
    avg_delivery_delay  = ('delivery_delay_days', 'mean'),
    last_purchase_date  = ('order_purchase_timestamp', 'max'),
    first_purchase_date = ('order_purchase_timestamp', 'min'),
).reset_index()

customer_features['days_since_last_purchase'] = (
    ref_date - customer_features['last_purchase_date']
).dt.days

customer_features['customer_lifetime_days'] = (
    customer_features['last_purchase_date'] -
    customer_features['first_purchase_date']
).dt.days

# CHURN = no purchase in last 180 days
customer_features['churned'] = (
    customer_features['days_since_last_purchase'] > 180
).astype(int)

print(f"Churn rate: {customer_features['churned'].mean()*100:.1f}%")

Churn rate: 58.9%


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

feature_cols = ['total_orders','total_revenue','avg_order_value',
                'avg_review_score','avg_delivery_delay',
                'days_since_last_purchase','customer_lifetime_days']

X = customer_features[feature_cols].fillna(0)
y = customer_features['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
auc    = roc_auc_score(y_test, y_prob)

print(f'AUC Score: {auc:.4f}')
print(classification_report(y_test, y_pred))

AUC Score: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7671
           1       1.00      1.00      1.00     11001

    accuracy                           1.00     18672
   macro avg       1.00      1.00      1.00     18672
weighted avg       1.00      1.00      1.00     18672



In [ ]:
X_all = customer_features[feature_cols].fillna(0)

customer_features['churn_probability'] = model.predict_proba(X_all)[:,1]

customer_features['churn_risk'] = pd.qcut(
    customer_features['churn_probability'],
    q=3,
    labels=['Low Risk','Medium Risk','High Risk']
)

high_risk = customer_features[customer_features['churn_risk'] == 'High Risk']
print(f"Revenue at risk: ${high_risk['total_revenue'].sum():,.0f}")

customer_features.to_csv('customer_churn_predictions.csv', index=False)

from google.colab import files
files.download('customer_churn_predictions.csv')

Revenue at risk: $3,879,087


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id     = '//ID//',
    aws_secret_access_key = '//password//',
    region_name           = 'us-east-1'
)

s3.upload_file(
    'customer_churn_predictions.csv',
    'reyaz-ecommerce-datalake-2026',
    'analytics/customer_churn_predictions.csv'
)
print('Uploaded to S3 successfully!')

Uploaded to S3 successfully!
